In [4]:
from sklearn.tree import DecisionTreeRegressor
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import inspect
from xgboost import XGBRegressor
import mlflow
import mlflow.sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

In [5]:
train_dataset=pd.read_csv('/home/aaic/Personal_Projects/Albany-Airbnb-Price-Prediction/central_data/feature_engineering/final_train_data.csv')
train_dataset.head()

,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,...,host_location_huizen,host_location_kingdom,host_location_london,host_location_netherlands,host_location_new,host_location_ny,host_location_paris,host_location_united,host_location_utrecht,host_location_york
0,2021,5,100,100,1,13.0,0,1,1,5.161503,...,0.0,0.0,0.0,0.690111,0.0,0.0,0.0,0.0,0.0,0.0
1,2025,5,100,90,0,4.0,0,1,1,5.061228,...,0.0,0.0,0.0,0.690111,0.0,0.0,0.0,0.0,0.0,0.0
2,2014,5,100,62,1,3.0,1,1,1,5.531524,...,0.0,0.0,0.0,0.690111,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,4,100,26,1,1.0,0,1,1,5.036768,...,0.0,0.0,0.0,0.690111,0.0,0.0,0.0,0.0,0.0,0.0
4,2013,5,100,50,0,3.0,0,1,1,5.361265,...,0.0,0.0,0.0,0.690111,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
train_dataset.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3642 entries, 0 to 3641
Data columns (total 95 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   host_since                                    3642 non-null   int64  
 1   host_response_time                            3642 non-null   int64  
 2   host_response_rate                            3642 non-null   int64  
 3   host_acceptance_rate                          3642 non-null   int64  
 4   host_is_superhost                             3642 non-null   int64  
 5   host_total_listings_count                     3642 non-null   float64
 6   host_verifications                            3642 non-null   int64  
 7   host_has_profile_pic                          3642 non-null   int64  
 8   host_identity_verified                        3642 non-null   int64  
 9   neighbourhood_cleansed                        3642 non-null   f

In [7]:
test_dataset=pd.read_csv('/home/aaic/Personal_Projects/Albany-Airbnb-Price-Prediction/central_data/feature_engineering/final_test_data.csv')
test_dataset.head()

,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,...,host_location_huizen,host_location_kingdom,host_location_london,host_location_netherlands,host_location_new,host_location_ny,host_location_paris,host_location_united,host_location_utrecht,host_location_york
0,2016,5,100,98,1,2.0,0,1,1,5.383628,...,0.0,0.0,0.0,1.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,2013,2,33,50,0,1.0,0,1,1,5.493564,...,0.0,0.0,0.0,0.690111,0.0,0.0,0.0,0.0,0.0,0.0
2,2012,1,100,0,0,1.0,0,1,1,5.061228,...,0.0,0.0,0.0,0.690111,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,3,70,67,0,1.0,2,1,1,5.219187,...,0.0,0.0,0.0,0.690111,0.0,0.0,0.0,0.0,0.0,0.0
4,2019,5,100,100,1,6.0,1,1,1,5.471066,...,0.0,0.0,0.0,0.690111,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# Data Preparation

X_train = train_dataset.drop('price', axis=1)
y_train = train_dataset['price']

X_test = test_dataset.drop('price', axis=1)
y_test = test_dataset['price']

In [9]:
mlflow.set_experiment("Algorithm Selection Experiment")  # All 3 runs will live under this experiment

# Run 1: Decision Tree 
with mlflow.start_run(run_name="Decision Tree"):
    # Define hyperparameters
    params = {"max_depth": 5, "min_samples_split": 10}
    
    # Log parameters BEFORE training
    mlflow.log_params(params)  # log_params() logs a dict at once
    
    # Train
    dt = DecisionTreeRegressor(**params)
    dt.fit(X_train, y_train)
    preds = dt.predict(X_test)
    
    # Log metrics AFTER training
    mlflow.log_metric("mean_absolute_error", mean_absolute_error(y_test, preds))
    mlflow.log_metric("mean_squared_error", mean_squared_error(y_test, preds))
    mlflow.log_metric('root_mean_squared_error', root_mean_squared_error(y_test, preds))
    
    # Save the model as an artifact
    mlflow.sklearn.log_model(dt, "decision_tree_model")

2026/05/15 09:55:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/15 09:55:34 INFO mlflow.store.db.utils: Updating database tables
2026/05/15 09:55:35 INFO mlflow.tracking.fluent: Experiment with name 'Algorithm Selection Experiment' does not exist. Creating a new experiment.
2026/05/15 09:55:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 09:55:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [10]:
# Run 2: Random Forest
with mlflow.start_run(run_name="Random Forest"):
    params = {"n_estimators": 60, "max_depth": 8, "min_samples_split": 5}
    
    mlflow.log_params(params)
    
    rf = RandomForestRegressor(**params)
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)
    
    mlflow.log_metric("mean_absolute_error", mean_absolute_error(y_test, preds))
    mlflow.log_metric("mean_squared_error", mean_squared_error(y_test, preds))
    mlflow.log_metric('root_mean_squared_error', root_mean_squared_error(y_test, preds))
    
    mlflow.sklearn.log_model(rf, "random_forest_model")

2026/05/15 09:55:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 09:55:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [11]:
# Run 3: XGBoost
with mlflow.start_run(run_name="XGBoost"):
    params = {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1, "subsample": 0.8}
    
    mlflow.log_params(params)
    
    xgb = XGBRegressor(**params)
    xgb.fit(X_train, y_train)
    preds = xgb.predict(X_test)
    
    mlflow.log_metric("mean_absolute_error", mean_absolute_error(y_test, preds))
    mlflow.log_metric("mean_squared_error", mean_squared_error(y_test, preds))
    mlflow.log_metric('root_mean_squared_error', root_mean_squared_error(y_test, preds))
    
    mlflow.xgboost.log_model(xgb, "xgboost_model")  # XGBoost has its own flavor

2026/05/15 09:55:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
